In [96]:
#Task 1 — Data Import & Setup

In [97]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

In [98]:
data=pd.read_csv("/content/social_media_engagement_5000.csv")

In [99]:
df = pd.DataFrame(data)

In [100]:
df.dtypes

,0
user_id,int64
age,float64
gender,object
country,object
post_id,int64
post_type,object
post_category,object
likes,float64
comments,float64
shares,float64


In [101]:
#change data type of posted at
df["posted_at"] = pd.to_datetime(df["posted_at"])

/tmp/ipykernel_5197/1958800760.py:2: UserWarning:

Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.



In [102]:
#Use NumPy arrays for basic array operations
df.head()
df.tail()
df.describe()
df.shape
df.columns
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   user_id           5000 non-null   int64         
 1   age               4850 non-null   float64       
 2   gender            4850 non-null   object        
 3   country           5000 non-null   object        
 4   post_id           5000 non-null   int64         
 5   post_type         5000 non-null   object        
 6   post_category     5000 non-null   object        
 7   likes             4850 non-null   float64       
 8   comments          4850 non-null   float64       
 9   shares            4850 non-null   float64       
 10  watch_time_sec    5000 non-null   int64         
 11  impression_count  5000 non-null   int64         
 12  posted_at         5000 non-null   datetime64[ns]
 13  follower_count    5000 non-null   int64         
 14  is_verified       5000 n

In [103]:
#Task 2 — Data Cleaning

In [104]:
df.isnull().sum()

,0
user_id,0
age,150
gender,150
country,0
post_id,0
post_type,0
post_category,0
likes,150
comments,150
shares,150


In [105]:
#removing null values from age column
df["age"]=df["age"].fillna(df["age"].median())

In [106]:
#removing null values from gender column
df["gender"]=df["gender"].fillna(df["gender"].mode()[0])

In [107]:
#removing null values from likes column
df["likes"]=df["likes"].fillna(df["likes"].median())

In [108]:
#removing null values from comments column
df["comments"]=df["comments"].fillna(df["comments"].median())

In [109]:
#removing null values from shares column
df["shares"]=df["shares"].fillna(df["shares"].median())

In [110]:
#removing null values from sentiment column
df["sentiment"]=df["sentiment"].fillna(df["sentiment"].mode()[0])

In [111]:
df.isna().sum()

,0
user_id,0
age,0
gender,0
country,0
post_id,0
post_type,0
post_category,0
likes,0
comments,0
shares,0


In [112]:
df.duplicated().sum()

np.int64(0)

In [113]:
df.dtypes

,0
user_id,int64
age,float64
gender,object
country,object
post_id,int64
post_type,object
post_category,object
likes,float64
comments,float64
shares,float64


In [114]:
#Fix incorrect data types
df["age"]=df["age"].astype(int)
df["likes"]=df["likes"].astype(int)
df["comments"]=df["comments"].astype(int)
df["shares"]=df["shares"].astype(int)

In [115]:
#Standardize column gender
df["gender"]=df["gender"].str.title()

In [116]:
#Correct unrealistic values in likes, comments, shares columns
df.loc[df["likes"]>100000] = df["likes"].median()
df.loc[df["comments"]>100000] = df["comments"].median()
df.loc[df["shares"]>100000] = df["shares"].median()

In [117]:
df.loc[df["likes"]<0] = df["likes"].median()
df.loc[df["comments"]<0] = df["comments"].median()
df.loc[df["shares"]<0] = df["shares"].median()

In [118]:
#Extract hashtag count
df["hashtag_count"]=df["hashtags"].str.count("#")
print(df["hashtag_count"])

0       3
1       1
2       1
3       3
4       1
       ..
4995    2
4996    2
4997    2
4998    3
4999    3
Name: hashtag_count, Length: 5000, dtype: int64


In [119]:
#Clean sentiment labels
df["sentiment"]=df["sentiment"].str.replace(";",",")

In [120]:
#Task 3 — Data Exploration using Pandas

In [121]:
df.head()
df.tail()
df.columns

Index(['user_id', 'age', 'gender', 'country', 'post_id', 'post_type',
       'post_category', 'likes', 'comments', 'shares', 'watch_time_sec',
       'impression_count', 'posted_at', 'follower_count', 'is_verified',
       'device_type', 'sentiment', 'hashtags', 'engagement_rate',
       'hashtag_count'],
      dtype='object')

In [122]:
df.dtypes
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   user_id           5000 non-null   int64         
 1   age               5000 non-null   int64         
 2   gender            5000 non-null   object        
 3   country           5000 non-null   object        
 4   post_id           5000 non-null   int64         
 5   post_type         5000 non-null   object        
 6   post_category     5000 non-null   object        
 7   likes             5000 non-null   int64         
 8   comments          5000 non-null   int64         
 9   shares            5000 non-null   int64         
 10  watch_time_sec    5000 non-null   int64         
 11  impression_count  5000 non-null   int64         
 12  posted_at         5000 non-null   datetime64[ns]
 13  follower_count    5000 non-null   int64         
 14  is_verified       5000 n

In [123]:
df["gender"].value_counts()
df["gender"].unique()
df["gender"].nunique()

3

In [124]:
#Create a correlation matrix for numeric fields.
corr_matrix = df.select_dtypes(include=[np.number]).corr()
print(corr_matrix)

                   user_id       age   post_id     likes  comments    shares  \
user_id           1.000000 -0.006688  0.020051  0.025811 -0.033395  0.013763   
age              -0.006688  1.000000 -0.013153 -0.036323 -0.007284  0.013871   
post_id           0.020051 -0.013153  1.000000  0.014526 -0.010540  0.001846   
likes             0.025811 -0.036323  0.014526  1.000000 -0.018421  0.004712   
comments         -0.033395 -0.007284 -0.010540 -0.018421  1.000000  0.006142   
shares            0.013763  0.013871  0.001846  0.004712  0.006142  1.000000   
watch_time_sec   -0.016847  0.005542  0.018374  0.008710 -0.016351  0.014658   
impression_count  0.015326  0.013322 -0.007709  0.007952 -0.009395 -0.005204   
follower_count    0.010124 -0.024894 -0.002844 -0.022982 -0.011733 -0.010783   
engagement_rate  -0.004282  0.008039  0.010139  0.093521  0.000051  0.021724   
hashtag_count    -0.013692  0.007173  0.007344 -0.002190 -0.015230  0.013379   

                  watch_time_sec  impre

In [125]:
df.groupby("post_type")["likes"].mean()

,likes
post_type,
image,10104.852446
reel,10037.784879
text,10100.132530
video,10188.586122


In [126]:
#Task 4 — Data Wrangling

In [127]:
#Use merge, concat, or join if combining DataFrames.
merged_df = pd.merge(df, df, on="post_id")
combained_df = pd.concat([df, df])
joined_df = df.join(df)

ValueError: columns overlap but no suffix specified: Index(['user_id', 'age', 'gender', 'country', 'post_id', 'post_type',
       'post_category', 'likes', 'comments', 'shares', 'watch_time_sec',
       'impression_count', 'posted_at', 'follower_count', 'is_verified',
       'device_type', 'sentiment', 'hashtags', 'engagement_rate',
       'hashtag_count'],
      dtype='object')

In [133]:
engagement_score = (df["likes"] + df["comments"] + df["shares"])
print(engagement_score)

0        8522
1       16163
2        6161
3        7482
4       16717
        ...  
4995    20060
4996    21241
4997    15151
4998    19896
4999    17131
Length: 5000, dtype: int64


In [147]:
df["log_likes"] = np.log1p(df["likes"])
df["log_comments"] = np.log1p(df["comments"])
df["log_shares"] = np.log1p(df["shares"])

In [148]:
#hashtag count
df["hashtag_count"]=df["hashtags"].str.count("#")
print(df["hashtag_count"])

0       3
1       1
2       1
3       3
4       1
       ..
4995    2
4996    2
4997    2
4998    3
4999    3
Name: hashtag_count, Length: 5000, dtype: int64


In [149]:
#groupby summaries by post_type
df.groupby("post_type")["likes"].mean()

,likes
post_type,
image,10104.852446
reel,10037.784879
text,10100.132530
video,10188.586122


In [150]:
#groupby summaries by country
df.groupby("country")["likes"].mean()

,likes
country,
Australia,10294.107505
Brazil,9886.678571
Canada,10063.054581
France,10372.862903
Germany,10130.434694
India,9962.444860
Japan,10088.851695
UAE,10227.473161
UK,9935.636917


In [151]:
#groupby summaries by sentiment
df.groupby("sentiment")["likes"].mean()

,likes
sentiment,
negative,10208.081250
neutral,9923.180092
positive,10180.046956


In [152]:
#Task 5 — Statistical Analysis

In [153]:
columns = [
    "likes",
    "comments",
    "shares",
    "watch_time_sec",
    "engagement_rate",
    "follower_count"
]

# Mean
print("MEAN")
print(df[columns].mean())

MEAN
likes               10106.982400
comments             1502.039800
shares               1002.910600
watch_time_sec       4014.503200
engagement_rate         0.964356
follower_count     393698.224800
dtype: float64


In [ ]:

# Median
print("MEDIAN")
print(df[columns].median())


In [ ]:

# Mode
print("MODE")
print(df[columns].mode().iloc[0])

In [ ]:
# Standard Deviation
print("STANDARD DEVIATION")
print(df[columns].std())

In [ ]:
# Variance
print("VARIANCE")
print(df[columns].var())

In [ ]:
# Percentiles
print(df[columns].quantile([0.25, 0.50, 0.75]))

In [ ]:
#Task 6 — Data Visualization (Min. 8 Plots Required)

In [ ]:
#Scatter: likes vs impressions
plt.scatter(df["impression_count"], df["likes"])
plt.show()

In [ ]:
#Line chart: daily engagement trend
df["posted_at"] = pd.to_datetime(df["posted_at"])
df.set_index("posted_at", inplace=True)
daily_engagement = df.resample("D").sum()["engagement_rate"]
plt.plot(daily_engagement)
plt.xlabel("Date")
plt.ylabel("Daily Engagement")
plt.title("Daily Engagement Trend")
plt.show()

In [ ]:
#Bar: posts by category
category_counts = df["post_type"].value_counts()
plt.bar(category_counts.index, category_counts.values)
plt.xlabel("Category")
plt.ylabel("Count")
plt.title("Posts by Category")
plt.show()

In [ ]:
#Pie: gender distribution
gender_counts = df["gender"].value_counts()
plt.pie(gender_counts.values, labels=gender_counts.index, autopct="%1.1f%%")
plt.title("Gender Distribution")
plt.show()

In [ ]:
#Histogram: age
plt.hist(df["age"], bins=10)
plt.xlabel("Age")
plt.ylabel("Frequency")
plt.title("Age Distribution")
plt.show()

In [ ]:
#Box: engagement rate
plt.boxplot(df["engagement_rate"])
plt.ylabel("Engagement Rate")
plt.title("Engagement Rate Distribution")
plt.show()

In [ ]:
#Seaborn

In [ ]:
# Count plot: post type
sns.countplot(data=df, x="post_type")
plt.show()

In [ ]:
# Bar plot: avg likes by category
sns.barplot(data=df, x="post_type", y="likes")
plt.show()

In [ ]:
#Violin: followers vs sentiment
sns.violinplot(data=df, x="sentiment", y="follower_count")
plt.show()

In [ ]:
#Pair plot: numeric features
num_cols = ["likes", "comments", "shares", "impression_count", "engagement_rate"]
sns.pairplot(df[num_cols])
plt.show()

In [ ]:
#Heatmap: correlation matrix
sns.heatmap(df[num_cols].corr())
plt.show()

In [ ]:
#Swarm plot: engagement vs device
sns.swarmplot(data=df.sample(200), x="device_type", y="engagement_rate")
plt.show()

In [128]:
#Plotly (Interactive)

In [129]:
#Line
px.line(df.reset_index(), x="posted_at", y="engagement_rate").show()

In [130]:
#Bar
px.bar(df.groupby("post_category")["likes"].mean().reset_index(),
    x="post_category",
    y="likes",
).show()

In [131]:
#Interactive Scatter Chart: likes vs impressions by category
px.scatter(df, x="impression_count", y="likes", color="post_type").show()

In [ ]:
#Final Insights should Include following analysis

In [ ]:
#Content Performance

In [134]:
post_type_eng = df.groupby("post_type")["engagement_rate"].mean()
print("Engagement by Post Type:", post_type_eng)

Engagement by Post Type: post_type
image    0.895646
reel     0.783047
text     1.064549
video    1.122365
Name: engagement_rate, dtype: float64


In [136]:
cat_eng = df.groupby("post_category")["engagement_rate"].mean()
print("Engagement by Category:", cat_eng)

Engagement by Category: post_category
education    0.844222
fashion      0.807109
fitness      0.865628
food         1.358628
lifestyle    1.091019
music        0.888803
tech         1.160475
travel       0.702223
Name: engagement_rate, dtype: float64


In [137]:
country_eng = df.groupby("country")["engagement_rate"].mean().sort_values(ascending=False)
print("Engagement by Country:", country_eng)

Engagement by Country: country
Brazil       1.540704
Australia    1.324339
France       1.146402
UAE          1.112352
Canada       0.916659
UK           0.850966
Japan        0.769623
Germany      0.759000
India        0.655005
USA          0.576580
Name: engagement_rate, dtype: float64


In [ ]:
# USER TRENDS

In [138]:
#How age affects engagement
age_eng = df.groupby("age")["engagement_rate"].mean()
print("Engagement by Age:", age_eng)

Engagement by Age: age
13    0.455583
14    1.003067
15    0.714170
16    1.226848
17    0.858763
18    0.888391
19    0.676687
20    0.515541
21    0.452752
22    0.586786
23    0.922448
24    2.897327
25    1.041681
26    0.501467
27    0.775328
28    0.385081
29    1.110432
30    1.270129
31    0.440684
32    0.581710
33    1.277557
34    0.557236
35    1.521596
36    0.609569
37    0.494109
38    1.035114
39    0.436184
40    0.930504
41    1.540070
42    1.350772
43    0.566439
44    0.795102
45    0.513521
46    0.663994
47    1.402015
48    1.675878
49    0.999235
50    1.552627
51    2.397295
52    0.867426
53    0.707924
54    1.002622
55    2.599106
56    0.524008
57    0.919751
58    0.435130
59    0.495578
60    0.759586
61    0.553854
62    1.005127
63    1.303909
64    0.402533
Name: engagement_rate, dtype: float64


In [140]:
#Performance difference for verified accounts
verified_eng = df.groupby("is_verified")["engagement_rate"].mean()
print("Engagement by Verification Status:", verified_eng)

Engagement by Verification Status: is_verified
False    0.954744
True     1.054250
Name: engagement_rate, dtype: float64


In [ ]:
#Behavioral Insights


In [141]:
#Best time of day for impressions
time_eng = df.groupby(df["posted_at"].dt.hour)["engagement_rate"].mean()
print("Engagement by Time of Day:", time_eng)

Engagement by Time of Day: posted_at
0    0.964356
Name: engagement_rate, dtype: float64


In [142]:
#Device type impact on watch time
device_eng = df.groupby("device_type")["watch_time_sec"].mean()
print("Engagement by Device Type:", device_eng)

Engagement by Device Type: device_type
desktop    3974.792521
mobile     4087.830760
tablet     3979.736429
Name: watch_time_sec, dtype: float64


In [ ]:
#Sentiment Analysis

In [143]:
#Which sentiment performs best
sentiment_eng = df.groupby("sentiment")["engagement_rate"].mean()
print("Engagement by Sentiment:", sentiment_eng)

Engagement by Sentiment: sentiment
negative    1.038486
neutral     0.991170
positive    0.919744
Name: engagement_rate, dtype: float64


In [146]:
#Behavior of negative/neutral sentiment posts
neg_neu_behavior = df.groupby("sentiment")[["likes", "comments", "shares", "watch_time_sec", "impression_count"]].mean()
print("\nDetailed Metrics by Sentiment:\n", neg_neu_behavior)


Detailed Metrics by Sentiment:
                   likes     comments       shares  watch_time_sec  \
sentiment                                                           
negative   10208.081250  1516.094792  1007.307292     4054.639583   
neutral     9923.180092  1528.404060   996.565160     4031.823183   
positive   10180.046956  1480.650617  1005.086749     3988.646240   

           impression_count  
sentiment                    
negative       50191.960417  
neutral        49515.971185  
positive       50248.107441  
